# LocStab: reproducible clustering benchmark

This is the corrected NUMBA benchmark.  Shared experiment logic lives in
`BenchmarkNumbaExperiments.py`, which prevents the LocPop and LocStab protocols
from drifting apart.

The run uses fixed data seeds, ten paired permutations, standardized vector
features, disjoint friendship/enmity relations, correctly named adjusted Rand
index (ARI), same-run warm starts, convergence diagnostics, and explicit
algorithm parameters.  It writes both raw and summarized CSVs and regenerates
individual plus four-panel summary plots.

Each clustering dataset is evaluated from singleton (S), predicted-$k$ (P),
$k$-means (KM), and DBSCAN (D) initializations.  The coalition-label capacity
is $n$, so every non-singleton partition permits a singleton-creation move
through an empty label.

Expected outputs:

- `csv/StableClustering/runs.csv`
- `csv/StableClustering/results.csv`
- `csv/StableClustering/preprocessing.csv`
- `csv/StableClustering/dataset-0.csv`, `dataset-1.csv`, `dataset-2.csv`
- `figures/StableClustering/*.png`


In [1]:
from importlib.metadata import version
from BenchmarkNumbaExperiments import run_clustering_experiment, DATA_SEED, THRESHOLDS, DOMAINS

REPETITIONS = 10
LOCAL_STABLE = True

print("Data seed:", DATA_SEED)
print("Repetitions:", REPETITIONS)
print("Thresholds:", THRESHOLDS)
print("Domains:", DOMAINS)

for package in ["numpy", "scipy", "pandas", "scikit-learn", "networkx", "numba", "matplotlib"]:
    print(f"{package}={version(package)}")


Data seed: 20260817
Repetitions: 10
Thresholds: ((0.2, 0.2), (0.25, 0.35), (0.4, 0.4))
Domains: ('B', 'AF', 'AE')
numpy=2.5.2
scipy=1.18.0
pandas=3.0.5
scikit-learn=1.9.0
networkx=3.6.1
numba=0.67.0
matplotlib=3.11.1


## Execute the complete experiment

This cell performs the full production run and overwrites the corresponding
CSV and figure artifacts.  Relationship preprocessing and the complete grid of
initializations can take some time.  Do not interrupt the
kernel while files are being written.


In [2]:
summary = run_clustering_experiment(local_stable=LOCAL_STABLE, repetitions=REPETITIONS)
summary

,Method,Dataset,Preference,Initialization,Beta Friend,Beta Enemy,Repetitions,Adjusted Rand Index,Adjusted Rand Index SD,Silhouette Score,...,Seconds,Seconds SD,Moves,Moves SD,Converged,Converged SD,Final Coalitions,Final Coalitions SD,Initial Adjusted Rand Index,Initial Adjusted Rand Index SD
0,KMeans,Moons,-,-,NaN,NaN,10,0.478969,5.551115e-17,0.487960,...,0.500916,1.046425,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DBSCAN,Moons,-,-,NaN,NaN,10,0.993355,0.000000e+00,0.017502,...,0.004248,0.001607,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,LocStab,Moons,B,S,0.2,0.2,10,0.227709,7.743995e-03,0.464808,...,0.128611,0.124190,364.8,9.239048,1.0,0.0,10.2,0.400000,0.000000,0.000000
3,LocStab,Moons,B,P,0.2,0.2,10,0.230663,5.921500e-03,0.438136,...,0.063817,0.010912,303.3,15.231874,1.0,0.0,10.6,0.800000,0.001426,0.002191
4,LocStab,Moons,B,KM,0.2,0.2,10,0.235948,2.053993e-04,0.465869,...,0.050777,0.010155,237.2,4.237924,1.0,0.0,10.0,0.000000,0.187561,0.000943
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147,LocStab,Iris,AF,D,0.4,0.4,10,0.000000,0.000000e+00,NaN,...,0.002571,0.000185,0.0,0.000000,1.0,0.0,1.0,0.000000,1.000000,0.000000
148,LocStab,Iris,AE,S,0.4,0.4,10,0.489232,3.322665e-02,0.247492,...,0.007976,0.000319,145.0,4.647580,1.0,0.0,7.3,0.781025,0.000000,0.000000
149,LocStab,Iris,AE,P,0.4,0.4,10,0.543252,4.866682e-02,0.277327,...,0.005377,0.001090,97.9,3.726929,1.0,0.0,5.5,0.500000,0.047471,0.015498
150,LocStab,Iris,AE,KM,0.4,0.4,10,0.609013,2.855242e-03,0.412953,...,0.002742,0.000207,4.8,0.600000,1.0,0.0,4.0,0.000000,0.916100,0.011469


## Verify convergence and output coverage

In [3]:
heuristic = summary[summary["Method"] == "LocStab"]
print("Summary rows:", len(summary))
print("Datasets:", sorted(summary["Dataset"].unique()))
print("Minimum convergence rate:", heuristic["Converged"].min())
print("Maximum recorded moves:", heuristic["Moves"].max())

if not (heuristic["Converged"] == 1.0).all():
    display(heuristic[heuristic["Converged"] < 1.0])
    raise RuntimeError("At least one run reached the move cap; do not use the outputs without investigation.")

display(summary.sort_values(["Dataset", "Method", "Initialization", "Preference"]).head(20))
print("Artifacts written under csv/StableClustering and figures/StableClustering")


Summary rows: 152
Datasets: ['3 Circles', 'Cancer', 'Iris', 'Moons']
Minimum convergence rate: 1.0
Maximum recorded moves: 579.4


,Method,Dataset,Preference,Initialization,Beta Friend,Beta Enemy,Repetitions,Adjusted Rand Index,Adjusted Rand Index SD,Silhouette Score,...,Seconds,Seconds SD,Moves,Moves SD,Converged,Converged SD,Final Coalitions,Final Coalitions SD,Initial Adjusted Rand Index,Initial Adjusted Rand Index SD
39,DBSCAN,3 Circles,-,-,NaN,NaN,10,0.187167,4.282094e-04,0.126546,...,0.003091,0.000111,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38,KMeans,3 Circles,-,-,NaN,NaN,10,0.444278,1.412614e-03,0.409868,...,0.067123,0.006792,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
51,LocStab,3 Circles,AE,D,0.20,0.20,10,0.240438,6.957058e-04,0.252134,...,0.022523,0.001397,174.0,0.000000,1.0,0.0,11.4,0.489898,0.435782,0.000829
63,LocStab,3 Circles,AE,D,0.25,0.35,10,0.289703,2.863003e-03,0.264502,...,0.015150,0.001104,107.7,0.781025,1.0,0.0,6.0,0.000000,0.535273,0.002394
75,LocStab,3 Circles,AE,D,0.40,0.40,10,0.398284,0.000000e+00,0.333045,...,0.018155,0.001088,133.8,0.979796,1.0,0.0,4.0,0.000000,0.408717,0.000436
47,LocStab,3 Circles,AF,D,0.20,0.20,10,0.284928,0.000000e+00,0.288281,...,0.013865,0.002664,83.6,0.800000,1.0,0.0,7.0,0.000000,0.683743,0.000810
59,LocStab,3 Circles,AF,D,0.25,0.35,10,0.403367,4.521932e-03,0.288868,...,0.019824,0.001122,153.6,0.489898,1.0,0.0,4.0,0.000000,0.286085,0.000561
71,LocStab,3 Circles,AF,D,0.40,0.40,10,0.180717,0.000000e+00,0.178953,...,0.027803,0.005203,213.4,0.489898,1.0,0.0,3.0,0.000000,0.141800,0.000148
43,LocStab,3 Circles,B,D,0.20,0.20,10,0.286226,3.720179e-03,0.343830,...,0.019643,0.000577,158.6,2.009975,1.0,0.0,7.6,0.489898,0.342355,0.005190
55,LocStab,3 Circles,B,D,0.25,0.35,10,0.417107,0.000000e+00,0.288812,...,0.021631,0.002178,158.0,1.095445,1.0,0.0,4.0,0.000000,0.279407,0.000380


Artifacts written under csv/StableClustering and figures/StableClustering
